# 00 — Setup & Basics

**What you'll learn:**
- What GenAI *evaluation* is and why it matters
- How the `azure-ai-evaluation` SDK is structured
- The dataset schema evaluators expect (`query`, `response`, `context`, `ground_truth`)
- Running your first `evaluate()` with a deterministic evaluator

---

## Why evaluate?

Shipping a prompt or model change without measuring quality is how apps regress silently.
**Evaluation** scores model outputs against test data on metrics such as relevance,
groundedness, fluency, and safety — turning "seems fine" into a number you can gate on.

The `azure-ai-evaluation` SDK gives you:

| Kind | Examples | Needs |
|------|----------|-------|
| **NLP / local** | F1, BLEU, ROUGE, METEOR, GLEU | nothing (deterministic) |
| **AI-assisted quality** | Relevance, Coherence, Fluency, Groundedness | a judge model |
| **Risk & safety** | Violence, HateUnfairness, ProtectedMaterial | an Azure AI Foundry project |
| **Custom** | your own code or LLM-judge evaluators | you define |

Everything runs through one entry point: `evaluate()`.

## Install & authenticate

```bash
cp .env.example .env      # fill in your Foundry + judge model details
uv sync                    # install azure-ai-evaluation and friends
az login                   # Entra ID auth — no keys needed for local NLP evaluators
```

In [1]:
import os
import site
import shutil
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# --------------------------------------------------------------------------
# Reasoning-model compatibility (gpt-5.x / o-series judge models)
# --------------------------------------------------------------------------
# Reasoning models reject `max_tokens` (they require `max_completion_tokens`) and
# only allow the default temperature. azure-ai-evaluation runs evaluators in
# spawned worker processes, so the fix must load in every worker: we copy the
# workshop's `sitecustomize.py` shim into site-packages (auto-imported by every
# Python process) and also apply it here in the kernel. It is a no-op for classic
# models (gpt-4o etc.), so it is always safe to run.
_shim_src = Path("../sitecustomize.py")
if _shim_src.exists():
    _shim_dst = Path(site.getsitepackages()[0]) / "sitecustomize.py"
    if not _shim_dst.exists() or _shim_dst.read_text() != _shim_src.read_text():
        shutil.copyfile(_shim_src, _shim_dst)
    exec(_shim_src.read_text())

# Judge model config for AI-assisted evaluators (notebooks 02, 04, 05).
# Uses az login / DefaultAzureCredential when AZURE_OPENAI_API_KEY is blank.
model_config = {
    "azure_endpoint": os.environ["AZURE_OPENAI_ENDPOINT"],
    "azure_deployment": os.environ["AZURE_OPENAI_DEPLOYMENT"],
    "api_version": os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
}
if os.environ.get("AZURE_OPENAI_API_KEY"):
    model_config["api_key"] = os.environ["AZURE_OPENAI_API_KEY"]

# Foundry project (notebooks 03, 06) — enables cloud logging + safety evaluators.
azure_ai_project = os.environ.get("AZURE_AI_PROJECT_ENDPOINT")

DATA_DIR = "../data"
print("Judge deployment:", model_config["azure_deployment"])

Judge deployment: gpt-5.4-mini


## The dataset schema

Evaluators read a **JSONL** file where each line is one test case. Common fields:

- `query` — the user input
- `response` — the model/app output being scored
- `context` — grounding text the answer should be based on (for groundedness/retrieval)
- `ground_truth` — the reference answer (for F1/BLEU/similarity)

Not every evaluator needs every field. Let's peek at the sample insurance dataset.

In [2]:
import pandas as pd

df = pd.read_json(f"{DATA_DIR}/insurance-qa.jsonl", lines=True)
df.head(3)

,query,response,context,ground_truth
0,How long do I have to file an auto claim after...,You should report an auto accident to your ins...,Policy terms: Auto claims must be filed within...,Auto claims must be filed within 30 days of th...
1,Does my homeowners policy cover water damage f...,Yes. Sudden and accidental water damage from a...,Homeowners HO-3: Covers sudden and accidental ...,"Yes, sudden burst-pipe water damage is covered..."
2,What is a deductible?,A deductible is the amount you pay out of pock...,Glossary: Deductible - the portion of a covere...,A deductible is the amount the policyholder pa...


## Your first evaluation

`F1ScoreEvaluator` is deterministic — it measures token overlap between `response`
and `ground_truth`, so it needs no judge model and costs nothing. It's the perfect
"hello world" for `evaluate()`.

In [3]:
from azure.ai.evaluation import evaluate, F1ScoreEvaluator

result = evaluate(
    data=f"{DATA_DIR}/insurance-qa.jsonl",
    evaluators={"f1": F1ScoreEvaluator()},
)

# Aggregated metrics across all rows:
result["metrics"]

2026-07-21 10:19:58 -0400 135608436917952 execution.bulk     INFO     Finished 8 / 8 lines.
2026-07-21 10:19:58 -0400 135608436917952 execution.bulk     INFO     Average execution time for completed lines: 0.0 seconds. Estimated time for incomplete lines: 0.0 seconds.


Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "f1_20260721_141958_325568"
Run status: "Completed"
Start time: "2026-07-21 14:19:58.325568+00:00"
Duration: "0:00:01.002595"

======= Combined Run Summary (Per Evaluator) =======

{
    "f1": {
        "status": "Completed",
        "duration": "0:00:01.002595",
        "completed_lines": 8,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    }
}




{'f1.f1_score': 0.49649287190471236,
 'f1.f1_score_score': 0.49649287190471236,
 'f1.f1_score_passed': 0.5,
 'f1.binary_aggregate': 0.5}

In [4]:
# Per-row scores live under result["rows"] — handy for spotting weak cases.
pd.DataFrame(result["rows"])[["inputs.query", "outputs.f1.f1_score"]]

,inputs.query,outputs.f1.f1_score
0,How long do I have to file an auto claim after...,0.375000
1,Does my homeowners policy cover water damage f...,0.606061
2,What is a deductible?,0.413793
3,Can I add my teenage child to my auto policy?,0.363636
4,Is flood damage covered under a standard homeo...,0.608696
5,How is my auto insurance premium calculated?,0.645161
6,What does liability coverage pay for?,0.777778
7,Will filing a claim raise my rates?,0.181818


Version mismatch between change and config responses.No configurations updated.
Version mismatch between change and config responses.No configurations updated.


## Recap

- Evaluation = scoring model outputs against test data on defined metrics.
- Data is JSONL with `query` / `response` / `context` / `ground_truth`.
- `evaluate(data=..., evaluators={...})` returns `metrics` (aggregate) and `rows` (per-case).

**Next:** [01 — NLP / local evaluators](./01-nlp-local-evaluators.ipynb) — more deterministic
metrics you can run for free.